## 3. Análisis Exploratorio de los Datos (EDA)

In [14]:
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Paths
preprocessed_input = Path("../data/processed/preprocessed_data.parquet")
refined_output = Path("../data/processed/refined_data.parquet")

#### 1) Revisión general

In [15]:
df = pd.read_parquet(preprocessed_input)
df.head()

,Dst Port,Protocol,Timestamp,Flow Duration,Tot Fwd Pkts,Tot Bwd Pkts,TotLen Fwd Pkts,TotLen Bwd Pkts,Fwd Pkt Len Max,Fwd Pkt Len Min,...,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label,source_file,time_seconds
0,0,0,2018-02-14 08:31:01,112641719,3,0,0,0,0,0,...,0.0,0,0,56320860.0,139.300034,56320958,56320761,Benign,Wednesday-14-02-2018_TrafficForML_CICFlowMeter,30661
1,0,0,2018-02-14 08:33:50,112641466,3,0,0,0,0,0,...,0.0,0,0,56320732.0,114.551300,56320814,56320652,Benign,Wednesday-14-02-2018_TrafficForML_CICFlowMeter,30830
2,0,0,2018-02-14 08:36:39,112638623,3,0,0,0,0,0,...,0.0,0,0,56319312.0,301.934601,56319525,56319098,Benign,Wednesday-14-02-2018_TrafficForML_CICFlowMeter,30999
3,22,6,2018-02-14 08:40:13,6453966,15,10,1239,2273,744,0,...,0.0,0,0,0.0,0.000000,0,0,Benign,Wednesday-14-02-2018_TrafficForML_CICFlowMeter,31213
4,22,6,2018-02-14 08:40:23,8804066,14,11,1143,2209,744,0,...,0.0,0,0,0.0,0.000000,0,0,Benign,Wednesday-14-02-2018_TrafficForML_CICFlowMeter,31223


In [16]:
# Size
print("Número de filas: ",df.shape[0])
print("Número de columnas: ",df.shape[1])

Número de filas:  3467793
Número de columnas:  82


In [19]:
df.dtypes.value_counts()
print(df.dtypes)
df['Protocol'] = df['Protocol'].astype('category')

Dst Port                 uint32
Protocol                  int64
Timestamp        datetime64[us]
Flow Duration             int64
Tot Fwd Pkts             uint32
                      ...      
Idle Max                  int64
Idle Min                  int64
Label                  category
source_file            category
time_seconds              int32
Length: 82, dtype: object


#### 2) Estudio de la variable objetivo

Ahora vamos a ver las diferentes clases que tenemos y en qué categorias se agrupan, pero para empezar necesitaremos crear el mapping the grupos tanto para ataque no ataque como para el tipo de ataque.

In [20]:
# --- Mapping for attack groups ---
attack_mapping = {
    'Benign': 'Benign',
    'DDOS attack-HOIC': 'DDOS',
    'DDOS attack-LOIC-UDP': 'DDOS',
    'DoS attacks-GoldenEye': 'DoS',
    'DoS attacks-Slowloris': 'DoS',
    'FTP-BruteForce': 'Bruteforce',
    'SSH-Bruteforce': 'Bruteforce',
    'Infilteration': 'Infiltration'
}

# New column identifies attack group
df['Attack_group'] = df['Label'].map(attack_mapping).astype('category')

# New column identifies attack or benign
df['Attack_or_Benign'] = df['Label'].apply(lambda x: 'Attack' if x != 'Benign' else 'Benign').astype('category')

Ahora pasaríamos a cres la primera tabla con el conteo y proporción de ataques vs no ataques. Las clases estan un poco desbalanceadas, 65% es trafico normal lo que indica que para nuestro analisis más adelante tendremos que usar técnicas que se adapten.

In [21]:
# --- Benign vs All Attacks ---
table_lvl1 = pd.DataFrame({
    'Count': df['Attack_or_Benign'].value_counts(),
    'Percentage': df['Attack_or_Benign'].value_counts(normalize=True) * 100
})
print("=== Tabla 1: Benign vs  Todos los ataques ===")
print(table_lvl1)

=== Tabla 1: Benign vs  Todos los ataques ===
                    Count  Percentage
Attack_or_Benign                     
Benign            2253976   64.997421
Attack            1213817   35.002579


Ahora pasamos a la tabla con los grupos de ataques, cabe destacar que los ataques más frecuentes son DDOS y Bruteforce que sabiendo un poco de la teoría de la ciberseguridad se conoce que estos se basan en el volumn de paquetes de ahí su frecuencia.

In [22]:
# --- Grouped Attack Categories ---
table_lvl2 = pd.DataFrame({
    'Count': df['Attack_group'].value_counts(),
    'Percentage': df['Attack_group'].value_counts(normalize=True) * 100
})
print("\n=== Tabla 2: Categorias de ataque agrupadas ===")
print(table_lvl2)


=== Tabla 2: Categorias de ataque agrupadas ===
                Count  Percentage
Attack_group                     
Benign        2253976   64.997421
DDOS           687742   19.832268
Bruteforce     380943   10.985171
Infiltration    92634    2.671267
DoS             52498    1.513874


Y ahora finalmente podemos ver con más detalle aún los diferentes tipos dentro de cada grupo.

In [23]:
# --- Detailed Attack Types (original) ---
table_lvl3 = pd.DataFrame({
    'Count': df['Label'].value_counts(),
    'Percentage': df['Label'].value_counts(normalize=True) * 100
})
print("\n=== Tabla 3: Tipos de ataque detallados ===")
print(table_lvl3)


=== Tabla 3: Tipos de ataque detallados ===
                         Count  Percentage
Label                                     
Benign                 2253976   64.997421
DDOS attack-HOIC        686012   19.782380
FTP-BruteForce          193354    5.575708
SSH-Bruteforce          187589    5.409464
Infilteration            92634    2.671267
DoS attacks-GoldenEye    41508    1.196957
DoS attacks-Slowloris    10990    0.316916
DDOS attack-LOIC-UDP      1730    0.049888


#### 3) Estudio de las variables numéricas

Para los graficos y representaciones me dirijo al rmd en R porque representa mucho mejor que en python. De todas formas realizaré un pequeño análiss de la distribución de las variables en esta tabla:

In [ ]:
df.describe()